# Single-chain tau-K32 analysis


## Setup

In [ ]:
import glob
import itertools
import os
import pickle
from collections import defaultdict

import matplotlib.pyplot as plt
import mdtraj as md
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap, PowerNorm
from scipy import stats

path_mod = "/home/tb/adlouet/DATA/PTM_LIBRARY/post_processed_trajectories/analysis/single_chain/"
path_unmod = path_mod + "unmod_tau/tauk32/transfer_charmm"
path_scripts = "/STORAGE/adlouet/DATA/PTM_LIBRARY/post_processed_trajectories/analysis/single_chain/scripts_fin/"
path_raw = "/home/tb/adlouet/DATA/PTM_LIBRARY/raw_trajectories/single_chain/"

modification_phosph = ['S202.T205', 'S258.262.356', 'full_phosph']
modification_acetly = ['280', '274.281', 'full_ace']
unmodified = ['unmodified']
all_systems = unmodified + modification_phosph + modification_acetly

panel_labels = {'unmodified': 'Unmodified', 'S202.T205': 'Ser202/Thr205', 'S258.262.356': 'Ser258/Ser262/Ser356',
                'full_phosph': 'All Ser/Thr', '280': 'Lys280', '274.281': 'Lys274/Lys281', 'full_ace': 'All Lys'}
first_resid = 198
modified_resnames = ['ALY', 'SP1', 'THZ', 'TP1']

fig_dir = 'figures_single_chain'
cache_dir = 'single_chain_cache'  
os.makedirs(fig_dir, exist_ok=True)
os.makedirs(cache_dir, exist_ok=True)

full_width = 7.0
sns.set_theme(style="ticks", context="paper", font="Liberation Sans", rc={
    'font.size': 10, 'axes.labelsize': 11, 'axes.titlesize': 11, 'xtick.labelsize': 10, 'ytick.labelsize': 10,
    'legend.fontsize': 10, 'axes.linewidth': 0.8, 'xtick.major.width': 0.8, 'ytick.major.width': 0.8,
    'mathtext.default': 'regular', 'pdf.fonttype': 42, 'ps.fonttype': 42,
})

def save_figure(fig, name):
    for ext in ['pdf', 'png']:
        fig.savefig(f'{fig_dir}/{name}.{ext}', dpi=600, bbox_inches=fig.bbox_inches)
    plt.show()

def cached(name, compute):
    path = f'{cache_dir}/{name}.pck'
    if os.path.exists(path):
        with open(path, 'rb') as file:
            return pickle.load(file)
    result = compute()
    with open(path, 'wb') as file:
        pickle.dump(result, file)
    return result

def replica_dirs(y):
    if y == 'unmodified':
        base = path_unmod
    elif y in modification_phosph:
        base = path_mod + f'phosph_tau/{y}'
    else:
        base = path_mod + f'acetyl_tau/{y}'
    return sorted(os.path.dirname(x) for x in glob.glob(base + '/rep_*/center.xtc'))

def load_replica(rep_dir):
    return md.load(rep_dir + '/center.xtc', top=rep_dir + '/charmm_box.gro')

def set_residue_ticks(ax, n_residues, step=50):
    resids = np.arange(200, first_resid + n_residues, step)
    ax.set_xticks(resids - first_resid, resids)
    ax.set_yticks(resids - first_resid, resids)

def compute_rg_series():
    return {y: [(traj.time / 1000, md.compute_rg(traj, masses=None) * 10)  # ps to ns, nm to A
                for traj in map(load_replica, replica_dirs(y))] for y in all_systems}

rg_by_replica = cached('rg_per_replica', compute_rg_series)
frame_dt_ns = np.median(np.diff(rg_by_replica['unmodified'][0][0]))
print(f'{frame_dt_ns * 1000:.0f} ps between stored frames')
for y, reps in rg_by_replica.items():
    print(f'{y:14s} {len(reps)} replicas, ' + ', '.join(f'{t[-1] / 1000:.2f}' for t, _ in reps) + ' us')

## Figure 1: cumulative charge along the sequence

In [ ]:
def read_cumulative_charge(topology):
    qtot = {}
    with open(topology) as f:
        for line in f:
            if '; residue' in line:
                resnum = int(line.split()[2])
            if 'qtot' in line:
                qtot[resnum + first_resid - 1] = float(line.split('qtot')[-1])
    charges = np.array(list(qtot.values())) - 1
    charges[-1] += 1
    return list(qtot), charges

charge_systems = [('Unmodified', 'unmod_tau/charmm/sample_reps', '#7570b3'),
                  ('Acetylated', 'acetyl_lib/full_ace', '#d95f02'),
                  ('Phosphorylated', 'phosph_lib/full_phosph', '#1b9e77')]

fig, ax = plt.subplots(figsize=(5.0, 3.3), constrained_layout=True)
ax.axhline(0, color='0.6', linewidth=1.0, linestyle='--')
for label, folder, color in charge_systems:
    resids, charges = read_cumulative_charge(f'{path_raw}{folder}/rep_0/pdb2gmx/charmm.top')
    ax.plot(resids, charges, color=color, linewidth=1.6, label=label)
    print(f'{label}: net charge {charges[-1]:+.0f}')
ax.set_xlabel('Residue number')
ax.set_ylabel('Cumulative charge (e)')
ax.set_xlim(198, 400)
ax.legend(frameon=False, loc='upper left')
sns.despine(ax=ax)
save_figure(fig, 'figure_1_cumulative_charge')

## Figure 4a,b: radius of gyration distributions

In [ ]:
rg_kde_frames = 1000

rg_palette = ['#332288', '#117733', '#ddcc77', '#88ccee']

def plot_rg_distributions(systems, title, name):
    fig, ax = plt.subplots(figsize=(3.5, 3.1), constrained_layout=True)
    for y, color in zip(systems, rg_palette):
        rg_flat = np.concatenate([rg[:rg_kde_frames] for _, rg in rg_by_replica[y]])
        sns.kdeplot(x=rg_flat, label=panel_labels[y], fill=False, linewidth=1.8, color=color, ax=ax)
    ax.set_title(title)
    ax.set_ylabel('Probability density')
    ax.set_xlabel(r'$R_G$ ($\AA$)')
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 0.06)
    ax.set_yticks([0, 0.02, 0.04, 0.06])
    sns.despine(ax=ax)
    fig.legend(*ax.get_legend_handles_labels(), loc='outside lower center', ncol=2, frameon=False,
               columnspacing=1.0, handlelength=1.4, fontsize=9.5)
    save_figure(fig, name)

plot_rg_distributions(modification_phosph + unmodified, 'Phosphorylated', 'figure_4a_rg_phosphorylated')
plot_rg_distributions(modification_acetly + unmodified, 'Acetylated', 'figure_4b_rg_acetylated')

## Figure S5: contact numbers at the modification sites

In [ ]:
contact_cutoff_nm = 0.8

def site_contact_counts(reps, sites, partners):
    pairs = list(itertools.product(sites, partners))
    counts = []
    for rep_dir in reps:
        traj = load_replica(rep_dir)
        distances, _ = md.compute_contacts(traj, contacts=pairs, scheme='ca', periodic=True)
        counts.append((distances.reshape(traj.n_frames, len(sites), len(partners)) < contact_cutoff_nm).sum(axis=2))
    return np.concatenate(counts)

def compute_site_contacts(y):
    top = md.load_topology(replica_dirs(y)[0] + '/charmm_box.gro')
    sites = [r.index for r in top.residues if r.name in modified_resnames]
    partners = [r.index for r in top.residues if r.name not in modified_resnames and r.is_protein]
    modified = site_contact_counts(replica_dirs(y), sites, partners)
    reference = site_contact_counts(replica_dirs('unmodified'), sites, partners)
    return [s + first_resid for s in sites], modified, reference

def contact_table(y):
    sites, modified, reference = cached(f'site_contacts_{y}', lambda: compute_site_contacts(y))
    return pd.concat(pd.DataFrame({'Site': site, 'Contacts': counts[:, i], 'Type': kind})
                     for i, site in enumerate(sites)
                     for counts, kind in [(modified, 'Modified'), (reference, 'Unmodified')])

def sites_in(y):
    return cached(f'site_contacts_{y}', lambda: compute_site_contacts(y))[0]

def plot_site_contacts(systems, name):
    fig = plt.figure(figsize=(full_width, 5.4), constrained_layout=True)
    grid = fig.add_gridspec(2, 2, width_ratios=[len(sites_in(systems[0])), len(sites_in(systems[1]))])
    axes = [fig.add_subplot(grid[0, 0]), fig.add_subplot(grid[0, 1]), fig.add_subplot(grid[1, :])]
    for ax, y in zip(axes, systems):
        sns.boxplot(data=contact_table(y), x='Site', y='Contacts', hue='Type', palette=['#3b3b8f', '#4aa8a0'], ax=ax,
                    width=0.75, linewidth=0.7, fliersize=1, flierprops={'rasterized': True, 'alpha': 0.4})
        ax.set_title(panel_labels[y])
        ax.set_xlabel('Modification site')
        ax.set_ylabel('Contact number')
        ax.set_ylim(0, 19)
        ax.set_yticks([0, 5, 10, 15])
        ax.get_legend().remove()
        sns.despine(ax=ax)
    axes[2].tick_params(axis='x', rotation=90)
    fig.legend(*axes[2].get_legend_handles_labels(), loc='outside upper right', ncol=2, frameon=False)
    save_figure(fig, name)

plot_site_contacts(modification_phosph, 'figure_s5a_contacts_phosphorylated')
plot_site_contacts(modification_acetly, 'figure_s5b_contacts_acetylated')

## Figure S6: Cα–Cα maps of the collapsed and extended states

In [ ]:
collapsed_rg = 30  # A

def normalized_mat(matrix):
    normalized_matrix = (matrix - matrix.min()) / (matrix.max() - matrix.min())
    normalized_matrix[np.isnan(normalized_matrix)] = 0
    return normalized_matrix

def compute_state_maps(y):
    collapsed, extended = [], []
    for rep_dir, (_, rg) in zip(replica_dirs(y), rg_by_replica[y]):
        traj = load_replica(rep_dir)
        distances, pairs = md.compute_contacts(traj, contacts='all', scheme='ca', periodic=True)
        square = md.geometry.squareform(distances, pairs)
        for state_maps, frames in [(collapsed, rg < collapsed_rg), (extended, rg >= collapsed_rg)]:
            if frames.any():
                state_maps.append(normalized_mat(square[frames].mean(axis=0)))
    return np.mean(collapsed, axis=0), np.mean(extended, axis=0)

def plot_state_maps(y, name):
    map_collapsed, map_extended = cached(f'state_maps_{y}', lambda: compute_state_maps(y))
    diff_map = map_collapsed - map_extended
    vmax = max(map_collapsed.max(), map_extended.max())
    fig, axes = plt.subplots(1, 3, figsize=(full_width, 2.6), constrained_layout=True, sharey=True)
    panels = [(map_collapsed, 'Collapsed', 'viridis', 0, vmax),
              (map_extended, 'Extended', 'viridis', 0, vmax),
              (diff_map, 'Collapsed − Extended', 'RdBu_r', -abs(diff_map).max(), abs(diff_map).max())]
    images = []
    for ax, (matrix, title, cmap, vmin, vmax_) in zip(axes, panels):
        images.append(ax.imshow(matrix, cmap=cmap, vmin=vmin, vmax=vmax_, origin='upper', interpolation='nearest'))
        ax.set_title(title)
        ax.set_xlabel('Residue')
        set_residue_ticks(ax, matrix.shape[0], step=100)
    axes[0].set_ylabel('Residue')
    fig.colorbar(images[0], ax=axes[:2], shrink=0.72, pad=0.02, label='Normalized distance')
    fig.colorbar(images[2], ax=axes[2], shrink=0.72, pad=0.02, label='Difference')
    save_figure(fig, name)

plot_state_maps('full_phosph', 'figure_s6a_state_maps_phosphorylated')
plot_state_maps('full_ace', 'figure_s6b_state_maps_acetylated')

## Figure S7: per-residue secondary structure

In [ ]:
color_E = '#231526'   # Sheet (E)
color_H = '#35a1ab'   # Helix (H)

def compute_dssp_propensity(y):
    dssp_all = np.vstack([md.compute_dssp(load_replica(rep_dir), simplified=True) for rep_dir in replica_dirs(y)])
    dssp_all = np.where(dssp_all == 'NA', 'C', dssp_all)
    return {state: (dssp_all == state).mean(axis=0) for state in 'EHC'}

def plot_secondary_structure(systems, name):
    fig, axes = plt.subplots(len(systems), 1, figsize=(full_width, 0.45 + 1.3 * len(systems)), sharex=True, sharey=True,
                             constrained_layout=True)
    for ax, y in zip(axes, systems):
        prop = cached(f'dssp_{y}', lambda: compute_dssp_propensity(y))
        print(y, ', '.join(f'{state} {prop[state].mean():.3f} +/- {prop[state].std():.3f}' for state in 'CHE'))
        residues = np.arange(len(prop['E'])) + first_resid
        ax.bar(residues, prop['E'], width=0.8, label='Sheet (E)', color=color_E, edgecolor='none')
        ax.bar(residues, prop['H'], width=0.8, bottom=prop['E'], label='Helix (H)', color=color_H, edgecolor='none')
        ax.set_title(panel_labels[y], loc='left', fontweight='bold', fontsize=10)
        ax.set_xlim(first_resid - 1, 401)
        ax.set_ylim(0, 0.6)
        ax.set_yticks([0, 0.2, 0.4, 0.6])
        sns.despine(ax=ax)
    axes[0].legend(frameon=False, loc='upper right', ncol=2, bbox_to_anchor=(1, 1.35))
    axes[-1].set_xlabel('Residue number')
    fig.supylabel('Secondary structure propensity', fontsize=11)
    save_figure(fig, name)

plot_secondary_structure(unmodified + modification_phosph, 'figure_s7a_secondary_structure_phosphorylated')
plot_secondary_structure(modification_acetly, 'figure_s7b_secondary_structure_acetylated')

## Figure S8: per-residue change in secondary structure

In [ ]:
ss_titles = {'E': 'Sheet (E)', 'H': 'Helix (H)', 'C': 'Coil (C)'}
ss_colors = {'E': color_E, 'H': color_H, 'C': '#aee3c0'}
negative_color = '#d1392b' 

def plot_secondary_structure_change(systems, name):
    reference = cached('dssp_unmodified', lambda: compute_dssp_propensity('unmodified'))
    residues = np.arange(len(reference['E'])) + first_resid
    fig, axes = plt.subplots(len(systems), len(ss_titles), figsize=(full_width, 5.0), sharex=True, sharey=True,
                             gridspec_kw={'hspace': 0.42, 'wspace': 0.08, 'left': 0.105, 'right': 0.99,
                                          'top': 0.87, 'bottom': 0.11})
    for row, y in enumerate(systems):
        prop = cached(f'dssp_{y}', lambda: compute_dssp_propensity(y))
        for ax, state in zip(axes[row], ss_titles):
            delta = reference[state] - prop[state]
            ax.bar(residues, delta, width=0.8, edgecolor='none',
                   color=[ss_colors[state] if d >= 0 else negative_color for d in delta])
            ax.axhline(0, color='black', linewidth=0.6)
            ax.set_xlim(first_resid - 1, 401)
            ax.set_xticks([200, 300, 400], ['200', '300', ''])
            ax.set_ylim(-0.5, 0.5)
            ax.set_yticks([-0.4, 0, 0.4], ['−0.4', '0', '0.4'])
            sns.despine(ax=ax)
        axes[row, 0].text(0, 1.06, panel_labels[y], transform=axes[row, 0].transAxes, fontweight='bold', va='bottom')
    for ax, title in zip(axes[0], ss_titles.values()):
        ax.text(0.5, 1.3, title, transform=ax.transAxes, ha='center')
    fig.supylabel(r'$\Delta$ propensity (unmodified $-$ modified)', x=0.008, fontsize=11)
    fig.supxlabel('Residue number', x=0.547, y=0.015, fontsize=11)
    save_figure(fig, name)

plot_secondary_structure_change(modification_phosph, 'figure_s8a_secondary_structure_change_phosphorylated')
plot_secondary_structure_change(modification_acetly, 'figure_s8b_secondary_structure_change_acetylated')

## Figure S9: residue-residue contact maps

In [ ]:
contact_map_cutoff_nm = 0.5

def compute_contact_frequency(y):
    total, n_frames = 0, 0
    for rep_dir in replica_dirs(y):
        traj = load_replica(rep_dir)
        distances, pairs = md.compute_contacts(traj, contacts='all', scheme='closest-heavy', periodic=True)
        total = total + (distances <= contact_map_cutoff_nm).sum(axis=0)
        n_frames += traj.n_frames
    frequency = md.geometry.squareform(total[np.newaxis], pairs)[0]
    near_diagonal = np.abs(np.subtract.outer(*[np.arange(len(frequency))] * 2)) < 3
    frequency[near_diagonal] = n_frames
    return normalized_mat(frequency.astype(float))

def load_contact_frequency(y, stored={}):
    if not stored:
        for pck in ['all_matrices_normalized_acetyl.pck', 'all_matrices_normalized_phosph.pck']:
            with open(path_scripts + pck, 'rb') as file:
                stored.update(pickle.load(file))
    legacy_labels = {'S202.T205': 'SER202 & THR205', 'S258.262.356': 'SER258 & SER262 & SER356',
                     'full_phosph': 'All SER & THR Sites'}
    key = legacy_labels.get(y, panel_labels[y])
    return stored[key] if key in stored else compute_contact_frequency(y)


fig, axes = plt.subplots(2, 4, figsize=(full_width, 3.9), sharex=True, sharey=True, constrained_layout=True)
layout = modification_acetly + unmodified + modification_phosph
for ax, y in zip(axes.flat, layout):
    matrix = load_contact_frequency(y)
    im = ax.imshow(matrix, cmap=LinearSegmentedColormap.from_list('white_blue', ['white', '#1f1fd6']), norm=PowerNorm(0.5, vmin=0, vmax=1),
                   origin='upper', interpolation='nearest')
    ax.set_title(panel_labels[y], fontsize=9.5 if len(panel_labels[y]) > 15 else 11)
    set_residue_ticks(ax, matrix.shape[0], step=100)
axes.flat[-1].remove()
fig.colorbar(im, ax=axes[:, -1].tolist()[:1]+axes[1, :-1].tolist(), fraction=0.03, pad=0.02,
             ticks=[0, 0.1, 0.25, 0.5, 0.75, 1], label='Normalized contact frequency')
fig.supxlabel('Residue', fontsize=11)
fig.supylabel('Residue', fontsize=11)
save_figure(fig, 'figure_s9_contact_maps')

## Figure S10: convergence of the radius of gyration
The table below is written to `figures_single_chain/figure_s10_rg_windows.csv`.

In [ ]:
window_ns = 200
min_replicas = 3

rows = []
for y, reps in rg_by_replica.items():
    for start in np.arange(0, max(t[-1] for t, _ in reps), window_ns):
        for rep_idx, (t, rg) in enumerate(reps):
            if t[-1] >= start + window_ns - frame_dt_ns:
                in_window = (t >= start) & (t < start + window_ns)
                rows.append({'system': y, 'start_ns': start, 'replica': rep_idx + 1,
                             'mean': rg[in_window].mean(), 'sd': rg[in_window].std(ddof=1)})
rg_windows = pd.DataFrame(rows)

rg_window_summary = rg_windows.groupby(['system', 'start_ns'], sort=False).agg(
    n_replicas=('mean', 'size'),
    mean_rg=('mean', 'mean'),
    sd_between_replicas=('mean', lambda x: x.std(ddof=1)),
    sem_between_replicas=('mean', 'sem'),
    mean_sd_within_replica=('sd', 'mean'),
).reset_index()
rg_window_summary = rg_window_summary[rg_window_summary['n_replicas'] >= min_replicas]
rg_window_summary.insert(2, 'end_ns', rg_window_summary['start_ns'] + window_ns)

replica_means = (rg_windows.pivot(index=['system', 'start_ns'], columns='replica', values='mean')
                 .add_prefix('mean_rg_replica_').reset_index())
rg_table = rg_window_summary.merge(replica_means, on=['system', 'start_ns'])
rg_table.to_csv(f'{fig_dir}/figure_s10_rg_windows.csv', index=False, float_format='%.2f')
rg_table.round(1)

In [ ]:
block_ns = 10 
raw_color, trend_color, grid_color = '#b8b8b8', '#332288', '#e6e6e6'
n_replicas = 5

def block_average(t, rg, block_ns):
    n_per_block = int(round(block_ns / frame_dt_ns))
    n_blocks = len(rg) // n_per_block
    return (t[:n_blocks * n_per_block].reshape(n_blocks, -1).mean(1),
            rg[:n_blocks * n_per_block].reshape(n_blocks, -1).mean(1))

fig, axes = plt.subplots(len(all_systems), n_replicas, figsize=(full_width, 9.0), sharex=True, sharey=True,
                         gridspec_kw={'hspace': 0.5, 'wspace': 0.05, 'left': 0.075, 'right': 0.995,
                                      'top': 0.92, 'bottom': 0.068})

for row, y in enumerate(all_systems):
    for rep_idx, (t, rg) in enumerate(rg_by_replica[y]):
        ax = axes[row, rep_idx]
        ax.plot(t / 1000, rg, color=raw_color, linewidth=0.4, rasterized=True)
        t_block, rg_block = block_average(t, rg, block_ns)
        ax.plot(t_block / 1000, rg_block, color=trend_color, linewidth=1.0)
        ax.set_xlim(0, 1.5)
        ax.set_xticks([0, 0.5, 1], ['0', '0.5', '1'])
        ax.set_ylim(15, 95)
        ax.set_yticks([20, 50, 80])
        ax.grid(axis='y', color=grid_color, linewidth=0.6)
        ax.set_axisbelow(True)
        sns.despine(ax=ax)
        if rep_idx > 0:
            ax.tick_params(axis='y', length=0)
            ax.spines['left'].set_visible(False)

    axes[row, 0].text(0, 1.08, panel_labels[y], transform=axes[row, 0].transAxes, fontweight='bold', va='bottom')

for rep_idx in range(n_replicas):
    axes[0, rep_idx].text(0.5, 1.5, f'Replica {rep_idx + 1}', transform=axes[0, rep_idx].transAxes, ha='center')

fig.supylabel(r'$R_G$ ($\AA$)', x=0.008, fontsize=11)
fig.supxlabel(r'Time ($\mu$s)', x=0.535, y=0.01, fontsize=11)
save_figure(fig, 'figure_s10_rg_convergence')

## Figure 5: salt bridges and phosphate–phosphate contacts

In [ ]:
contact_definition = 'submitted'

phosphate_contact_selections = {
    'submitted': ("name O1P or name O2P or name OJ", "resn LYS or resn ARG and name NZ NH1 NH2 NE"),
    'side_chain': ("name O1P or name O2P or name OJ or name OT or (resn TP1 and name O2 O3 O4)",
                   "(resn LYS and name NZ) or (resn ARG and name NH1 NH2 NE)"),
}
phosphate_cutoff_nm = 0.40

def mean_partner_residues(traj, atom_pairs, sites):
    top = traj.topology
    in_contact = md.compute_distances(traj, atom_pairs) < phosphate_cutoff_nm
    pairs_by_residue = defaultdict(list)
    for idx, (i, j) in enumerate(atom_pairs):
        pairs_by_residue[(top.atom(i).residue.index, top.atom(j).residue.index)].append(idx)
    counts = dict.fromkeys(sites, 0.0)
    for residue_pair, idxs in pairs_by_residue.items():
        occupancy = in_contact[:, idxs].any(axis=1).mean()
        for residue in residue_pair:
            if residue in counts:
                counts[residue] += occupancy
    return [counts[site] for site in sites]

def compute_phosphate_contacts(phosphate_selection, basic_selection):
    phosphate_phosphate, salt_bridge = [], []
    for rep_dir in replica_dirs('full_phosph'):
        traj = load_replica(rep_dir)
        top = traj.topology
        group_phosphate = top.select(phosphate_selection)
        group_basic = top.select(basic_selection)
        sites = list(dict.fromkeys(top.atom(i).residue.index for i in group_phosphate))
        pairs_ph_ph = [(i, j) for i, j in itertools.combinations(group_phosphate, 2)
                       if top.atom(i).residue.index != top.atom(j).residue.index]
        phosphate_phosphate.append(mean_partner_residues(traj, pairs_ph_ph, sites))
        salt_bridge.append(mean_partner_residues(traj, list(itertools.product(group_phosphate, group_basic)), sites))
    return [site + first_resid for site in sites], np.mean(phosphate_phosphate, axis=0), np.mean(salt_bridge, axis=0)

sites, mean_ph_ph, mean_salt_bridge = cached(f'phosphate_contacts_{contact_definition}', lambda: compute_phosphate_contacts(
    *phosphate_contact_selections[contact_definition]))

x = np.arange(len(sites))
width = 0.38
fig, ax = plt.subplots(figsize=(full_width, 3.1), constrained_layout=True)
ax.bar(x - width / 2, mean_ph_ph, width, label='Phosphate\u2013phosphate', color=rg_palette[0])
ax.bar(x + width / 2, mean_salt_bridge, width, label='Salt bridge (Lys/Arg)', color=rg_palette[1])
ax.set_xticks(x, sites, rotation=90)
ax.set_xlim(-0.7, len(sites) - 0.3)
ax.set_xlabel('Phosphorylated residue')
ax.set_ylabel('Mean contacts per frame')
sns.despine(ax=ax)
fig.legend(*ax.get_legend_handles_labels(), loc='outside upper right', ncol=2, frameon=False)
save_figure(fig, 'figure_5_phosphate_contacts')

## Figure 6: Rg against SCD and κ

In [ ]:
topology_dirs = {'280': 'acetyl_lib/280', '274.281': 'acetyl_lib/274.281', 'full_ace': 'acetyl_lib/full_ace',
                 'S202.T205': 'phosph_lib/S202.T205.S208', 'S258.262.356': 'phosph_lib/S258.262.356',
                 'full_phosph': 'phosph_lib/full_phosph', 'unmodified': 'unmod_tau/charmm/sample_reps'}
aa_map = {'ALA': 'A', 'ARG': 'R', 'ASN': 'N', 'ASP': 'D', 'CYS': 'C', 'GLN': 'Q', 'GLU': 'E', 'GLY': 'G', 'ILE': 'I',
          'LEU': 'L', 'LYS': 'K', 'MET': 'M', 'PHE': 'F', 'PRO': 'P', 'SER': 'S', 'THR': 'T', 'TRP': 'W', 'TYR': 'Y',
          'VAL': 'V', 'HIS': 'H', 'HSD': 'H', 'HSE': 'H', 'HSP': 'H'}

def read_sequence(y, modified_letters):
    resnames = []
    with open(f'{path_raw}{topology_dirs[y]}/rep_0/pdb2gmx/charmm.top') as f:
        for line in f:
            if '; residue' in line:
                resname = line.split()[3]
            if 'qtot' in line:
                resnames.append(resname)
    return ''.join({**aa_map, **modified_letters}.get(name, 'X') for name in resnames)

charge_map = {'K': +1, 'R': +1, 'D': -1, 'E': -1, 'X': -1}

def compute_scd(sequence):
    # SCD = (1/N) * sum_i sum_j<i q_i * q_j * sqrt(i - j)
    q = np.array([charge_map.get(aa, 0) for aa in sequence])
    return sum(q[i] * q[j] * np.sqrt(i - j) for i in range(1, len(q)) for j in range(i)) / len(q)

scd = {y: compute_scd(read_sequence(y, {'SP1': 'X', 'THZ': 'X', 'TP1': 'X', 'ALY': 'Y'})) for y in topology_dirs}

for y in topology_dirs:
    print(f">{y}\n{read_sequence(y, {'SP1': 'D', 'THZ': 'D', 'TP1': 'D', 'ALY': 'T'})}")
kappa = {'274.281': 0.11334, '280': 0.11066, 'full_ace': 0.16755, 'S202.T205': 0.11901, 'S258.262.356': 0.10887,
         'full_phosph': 0.09185, 'unmodified': 0.11087}


correlation_systems = modification_phosph + modification_acetly
mean_rg = {y: np.concatenate([rg[:rg_kde_frames] for _, rg in rg_by_replica[y]]).mean() for y in correlation_systems}

markers = {y: (color, marker) for systems, marker in [(modification_phosph, 'o'), (modification_acetly, 's')]
           for y, color in zip(systems, rg_palette)}

rg_vals = np.array([mean_rg[y] for y in correlation_systems])
fig, axes = plt.subplots(1, 2, figsize=(full_width, 3.6), constrained_layout=True)
for ax, values, ylabel, text_x, text_ha in [(axes[0], scd, 'SCD', 0.04, 'left'), (axes[1], kappa, r'$\kappa$', 0.96, 'right')]:
    yvals = np.array([values[y] for y in correlation_systems])
    for y in correlation_systems:
        color, marker = markers[y]
        ax.scatter(mean_rg[y], values[y], s=50, color=color, marker=marker, edgecolors='0.25', linewidths=0.5,
                   zorder=3, label=panel_labels[y])
    fit = stats.linregress(rg_vals, yvals)
    x_fit = np.array([rg_vals.min(), rg_vals.max()])
    ax.plot(x_fit, fit.slope * x_fit + fit.intercept, color='0.4', linestyle='--', linewidth=1.0)
    ax.text(text_x, 0.97, f'$r$ = {fit.rvalue:.3f}, $p$ = {fit.pvalue:.3f}', transform=ax.transAxes, ha=text_ha, va='top')
    ax.set_xlabel(r'$R_G$ ($\AA$)')
    ax.set_ylabel(ylabel)
    sns.despine(ax=ax)
fig.legend(*axes[0].get_legend_handles_labels(), loc='outside lower center', ncol=3, frameon=False, columnspacing=1.5)
save_figure(fig, 'figure_6_rg_vs_charge_descriptors')

pd.DataFrame({'Rg (A)': mean_rg, 'SCD': {y: scd[y] for y in correlation_systems}, 'kappa': {y: kappa[y] for y in correlation_systems}}).rename(index=panel_labels).round(3)